In [ ]:
import glob
import os
import subprocess
import moviepy
import random

def parse_gesture_file(gesture_file):
    """Parse the gesture annotation file and return list of actions"""
    actions = []
    with open(gesture_file, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 4:
                try:
                    start_frame = int(parts[2]) / 1000.0  # Convert to seconds
                    end_frame = int(parts[3]) / 1000.0    # Convert to seconds
                    gesture_type = parts[4] if len(parts) > 4 else "Unknown"
                    
                    if end_frame > start_frame:
                        actions.append({
                            'start_time': start_frame,
                            'end_time': end_frame,
                            'type': gesture_type
                        })
                except ValueError as e:
                    print(f"Error parsing line: {line.strip()}, Error: {e}")
                    continue
    return actions

def get_nogesture_intervals(actions, total_duration):
    """Extract intervals without gestures"""
    nogesture_intervals = []
    last_end = 0
    
    # Sort actions by start time
    sorted_actions = sorted(actions, key=lambda x: x['start_time'])
    
    for action in sorted_actions:
        if action['start_time'] > last_end and action['start_time'] < total_duration:
            nogesture_intervals.append({
                'start_time': last_end,
                'end_time': min(action['start_time'], total_duration)
            })
        last_end = action['end_time']
    
    if last_end < total_duration:
        nogesture_intervals.append({
            'start_time': last_end,
            'end_time': total_duration
        })
    
    return nogesture_intervals

def find_random_nogesture_segment(nogesture_intervals, desired_duration):
    """Find a random segment within nogesture intervals matching the desired duration"""
    # Filter intervals that are long enough
    valid_intervals = [interval for interval in nogesture_intervals 
                      if (interval['end_time'] - interval['start_time']) >= desired_duration]
    
    if not valid_intervals:
        return None
    
    # Pick a random interval
    interval = random.choice(valid_intervals)
    
    # Calculate the maximum start time that allows for the desired duration
    max_start = interval['end_time'] - desired_duration
    
    # Pick a random start time within the valid range
    random_start = random.uniform(interval['start_time'], max_start)
    
    return {
        'start_time': random_start,
        'end_time': random_start + desired_duration
    }

def extract_clip(input_file, output_file, start_time, end_time):
    """Extract clip using ffmpeg with re-encoding"""
    try:
        duration = end_time - start_time
        
        cmd = [
            'ffmpeg', '-y',
            '-ss', str(start_time),
            '-i', input_file,
            '-t', str(duration),
            '-c:v', 'libx264',
            '-c:a', 'aac',
            '-strict', 'experimental',
            output_file
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if os.path.exists(output_file) and os.path.getsize(output_file) > 1024:
            return True
        else:
            print(f"Failed to create valid output file: {output_file}")
            print(f"FFmpeg output: {result.stderr}")
            if os.path.exists(output_file):
                os.remove(output_file)
            return False
            
    except Exception as e:
        print(f"Error extracting clip: {e}")
        if os.path.exists(output_file):
            os.remove(output_file)
        return False

def process_video(video_path, output_folder):
    """Process a single video file"""
    gesture_folder = os.path.join(output_folder, 'gestures')
    nogesture_folder = os.path.join(output_folder, 'nogestures')
    moves_folder = os.path.join(output_folder, 'move')
    os.makedirs(gesture_folder, exist_ok=True)
    os.makedirs(nogesture_folder, exist_ok=True)
    os.makedirs(moves_folder, exist_ok=True)
    
    video_id = os.path.splitext(os.path.basename(video_path))[0]
    gesture_file = os.path.join(os.path.dirname(video_path), f"{video_id}.txt")
    
    if not os.path.exists(gesture_file):
        print(f"No gesture file found for {video_path}")
        return
    
    try:
        # Get video duration
        video = moviepy.VideoFileClip(video_path)
        total_duration = video.duration
        video.close()
        
        print(f"Video duration: {total_duration:.2f} seconds")
        
        # Load and validate actions
        actions = parse_gesture_file(gesture_file)
        valid_actions = [a for a in actions if a['end_time'] <= total_duration and a['start_time'] >= 0]
        
        # Get non-gesture intervals
        nogesture_intervals = get_nogesture_intervals(valid_actions, total_duration)
        
        # Separate gestures and moves
        gestures = [a for a in valid_actions if a['type'].upper() != 'N/A']
        moves = [a for a in valid_actions if a['type'].upper() == 'N/A']
        
        # Process regular gesture clips and matching non-gesture clips
        for idx, action in enumerate(gestures):
            gesture_duration = action['end_time'] - action['start_time']
            
            # Process gesture clip
            safe_type = "".join(c if c.isalnum() else "_" for c in action["type"])
            gesture_output = os.path.join(
                gesture_folder, 
                f'gesture_{video_id}_{idx:04d}_{safe_type}.mp4'
            )
            
            if not os.path.exists(gesture_output):
                print(f"Extracting gesture clip {idx} from {video_id}: "
                      f"{action['start_time']:.2f}s - {action['end_time']:.2f}s "
                      f"(Type: {action['type']})")
                      
                if extract_clip(video_path, gesture_output, action['start_time'], action['end_time']):
                    print(f"Successfully extracted gesture clip: {gesture_output}")
                    
                    # Find and extract matching non-gesture clip
                    nogesture_segment = find_random_nogesture_segment(nogesture_intervals, gesture_duration)
                    if nogesture_segment:
                        nogesture_output = os.path.join(
                            nogesture_folder, 
                            f'nogesture_{video_id}_{idx:04d}.mp4'
                        )
                        
                        print(f"Extracting matching non-gesture clip {idx} from {video_id}: "
                              f"{nogesture_segment['start_time']:.2f}s - {nogesture_segment['end_time']:.2f}s")
                              
                        if extract_clip(video_path, nogesture_output, 
                                      nogesture_segment['start_time'], 
                                      nogesture_segment['end_time']):
                            print(f"Successfully extracted non-gesture clip: {nogesture_output}")
                        else:
                            print(f"Failed to extract non-gesture clip: {nogesture_output}")
                    else:
                        print(f"Could not find suitable non-gesture interval of duration {gesture_duration:.2f}s")
                else:
                    print(f"Failed to extract gesture clip: {gesture_output}")

        # Process move clips (N/A gestures)
        for idx, action in enumerate(moves):
            output_file = os.path.join(
                moves_folder, 
                f'move_{video_id}_{idx:04d}.mp4'
            )
            
            if not os.path.exists(output_file):
                print(f"Extracting move clip {idx} from {video_id}: "
                      f"{action['start_time']:.2f}s - {action['end_time']:.2f}s")
                      
                if extract_clip(video_path, output_file, action['start_time'], action['end_time']):
                    print(f"Successfully extracted move clip: {output_file}")
                else:
                    print(f"Failed to extract move clip: {output_file}")
                
    except Exception as e:
        print(f"Error processing video {video_path}: {e}")

def main():
    input_folder = 'G:/gesture_datasets/Multisimo/videos_annotations/'
    output_folder = 'G:/gesture_datasets/Multisimo/trainingvideos/'
    
    video_extensions = ['.mp4', '.avi', '.mov']
    video_list = []
    for ext in video_extensions:
        video_list.extend(glob.glob(os.path.join(input_folder, f'*{ext}')))
    
    for video_path in video_list:
        print(f"\nProcessing video: {video_path}")
        process_video(video_path, output_folder)

if __name__ == "__main__":
    main()